In [29]:
#imports
import matplotlib.pyplot as plt
import os
import pandas as pd
import numpy as np
import joblib
from datetime import datetime
import random

# Set the random seed for reproducibility
seed=42
np.random.seed(seed)

data_PATH = "C:\\Users\\andre\\PhD\Datasets\\iam online"
image_PATH=data_PATH+"\\unzipped"
output_path=".\\datasets"
if not os.path.exists(output_path):
    os.makedirs(output_path)
dataset_name = "iam"

# run

In [30]:
files = get_all_file_paths(os.path.join(data_PATH, "lineImages-all", "lineImages"))
#print(files[0:2])
base_names = [os.path.splitext(os.path.basename(f))[0] for f in files]
#print(base_names[0:2])
df_offline = pd.DataFrame({
    'file_path_offline': files,
    'id': base_names
})
print(list(df_offline.iloc[0]))

files = get_all_file_paths(os.path.join(data_PATH, "lineStrokes-all", "lineStrokes"))
#print(files[0:2])
base_names = [os.path.splitext(os.path.basename(f))[0] for f in files]
#print(base_names[0:2])
df_online = pd.DataFrame({
    'file_path_online': files,
    'id': base_names
})
print(list(df_online.iloc[0]))

print(len(df_offline), len(df_online))
df_merged = pd.merge(df_offline, df_online, on='id',how='outer', suffixes=('_offline', '_online'))
display(df_merged.head())
print(len(df_merged))

df_merged['form_id'] = df_merged['id'].str.replace(r'-\w{2}$', '', regex=True)
df_merged[['id', 'form_id']].head()

txt_file=os.path.join(data_PATH, "forms.txt")
with open(txt_file, 'r') as f:
    lines = f.readlines()
lines = [line.strip() for line in lines if line.strip()]
lines = [line for line in lines if not line.startswith('#')]
form_id_list = [line.split(' ')[0] for line in lines]
writer = [line.split(' ')[1] for line in lines]
device = [line.split(' ')[2] for line in lines]
df_forms = pd.DataFrame({
    'form_id': form_id_list,
    'writer': writer,
    'device': device
})
df_forms.head()
df_merged = df_merged.merge(df_forms, on='form_id', how='left')
print(len(df_merged))

['C:\\Users\\andre\\PhD\\Datasets\\iam online\\lineImages-all\\lineImages\\a01\\a01-000\\a01-000u-01.tif', 'a01-000u-01']
['C:\\Users\\andre\\PhD\\Datasets\\iam online\\lineStrokes-all\\lineStrokes\\a01\\a01-000\\a01-000u-01.xml', 'a01-000u-01']
13017 12195


,file_path_offline,id,file_path_online
0,C:\Users\andre\PhD\Datasets\iam online\lineIma...,a01-000u-01,C:\Users\andre\PhD\Datasets\iam online\lineStr...
1,C:\Users\andre\PhD\Datasets\iam online\lineIma...,a01-000u-02,C:\Users\andre\PhD\Datasets\iam online\lineStr...
2,C:\Users\andre\PhD\Datasets\iam online\lineIma...,a01-000u-03,C:\Users\andre\PhD\Datasets\iam online\lineStr...
3,C:\Users\andre\PhD\Datasets\iam online\lineIma...,a01-000u-04,C:\Users\andre\PhD\Datasets\iam online\lineStr...
4,C:\Users\andre\PhD\Datasets\iam online\lineIma...,a01-000u-05,C:\Users\andre\PhD\Datasets\iam online\lineStr...


13025
13025


In [31]:
#examine form_id, id and writer fields
df_merged[df_merged['form_id']=='a01-000x']['writer']
df_temp=df_merged[df_merged['writer']=='10011']
#get unique ids for the writer
unique_ids = df_temp['id'].unique()
print(unique_ids)

['a01-000u-01' 'a01-000u-02' 'a01-000u-03' 'a01-000u-04' 'a01-000u-05'
 'a01-000u-06' 'a01-014x-00' 'a01-014x-01' 'a01-014x-02' 'a01-014x-03'
 'a01-014x-04' 'a01-053x-01' 'a01-053x-02' 'a01-053x-03' 'a01-053x-04'
 'a01-053x-05' 'a01-053x-06' 'a01-053x-07' 'a02-004-01' 'a02-004-02'
 'a02-004-03' 'a02-004-04' 'a02-004-05' 'a02-004-06' 'b04-187-00'
 'b04-187-01' 'b04-187-02' 'b04-187-03' 'b04-187-04' 'c02-007-01'
 'c02-007-02' 'c02-007-03' 'c02-007-04' 'c02-007-05' 'c02-007-06'
 'c02-007-07' 'c04-023-00' 'c04-023-01' 'c04-023-02' 'c04-023-03'
 'c04-023-04' 'c04-080-00' 'c04-080-01' 'c04-080-02' 'c04-080-03'
 'c04-080-04' 'd05-013-01' 'd05-013-02' 'd05-013-03' 'd05-013-04'
 'd05-013-05' 'd05-013-06' 'd05-013-07' 'd05-013-08' 'e02-100-00'
 'e02-100-01' 'e02-100-02' 'e02-100-03' 'e02-100-04' 'f04-074-00'
 'f04-074-01' 'f04-074-02' 'f04-074-03' 'f04-074-04' 'g07-038-01'
 'g07-038-02' 'g07-038-03' 'g07-038-04' 'g07-038-05' 'g07-038-06'
 'g07-065-01' 'g07-065-02' 'g07-065-03' 'g07-065-04' 'g07-

In [32]:
#number of unique writers
num_unique_writers = df_merged['writer'].nunique()
print(num_unique_writers)
from lxml import etree
xml_file=os.path.join(data_PATH, "writers.xml")
# Parse with lxml
tree = etree.parse(xml_file)
root = tree.getroot()
# Extract data
records = []
for writer in root.findall('Writer'):
    data = dict(writer.attrib)

    # Get all <Science> and <WrittenLanguage> elements as lists
    data['Sciences'] = [s.text for s in writer.findall('Science')]
    data['WrittenLanguages'] = [w.text for w in writer.findall('WrittenLanguage')]

    records.append(data)
# Create DataFrame
df_writers = pd.DataFrame(records)
print(df_writers.head())
df_merged = df_merged.merge(df_writers, left_on='writer', right_on='name', how='left')

217
    name  DayOfBirth EducationalDegree Gender  NativeCountry NativeLanguage  \
0  10000  1982-06-08     Dipl. Inform.   Male        Germany         German   
1  10001                                 Male         France         French   
2  10002                                 Male  Great Britain        English   
3  10003                                 Male         France         French   
4  10004                                 Male         France         French   

  OtherLanguage   Profession   WritingType                         Sciences  \
0        French  PhD-Student  Right-handed  [Computer Science, Mathematics]   
1           NaN          NaN  Right-handed                               []   
2           NaN          NaN  Right-handed                               []   
3           NaN          NaN  Right-handed                               []   
4           NaN          NaN  Right-handed                               []   

    WrittenLanguages WrittenLanguage Science  

In [33]:
#get unique values for Gender
unique_genders = df_merged['Gender'].unique()
print(unique_genders)
#set the gender column to 1 for male and 0 for female, remove rows with unknown gender
df_merged = df_merged[df_merged['Gender'].isin(['Male', 'Female'])]
df_merged['male'] = df_merged['Gender'].apply(lambda x: 1 if x == 'Male' else 0)
unique_genders = df_merged['male'].unique()
print(unique_genders)

['Male' 'Female' nan]
[1 0]


In [34]:
#filter and rename columns for compatibility with the icdar code
df_merged = df_merged.rename(columns={'file_path_offline': 'file_name'})
df_merged['isEng']=0
#assign a different integer to each different value in the form_id column, call the new categorical column 'same_text'
df_merged['same_text'] = df_merged['form_id'].astype('category').cat.codes
df_merged = df_merged[['file_name', 'writer', 'male', 'isEng', 'same_text','form_id']]
#check if there are -1 values in the same_text column
if (df_merged['same_text'] == -1).any():
    print("There are -1 values in the same_text column.")


In [35]:
#check if there are nan values in the file_name column
print(f"Number of NaN values in file_name column: {df_merged['file_name'].isnull().sum()}")
#drop rows with nan values in the file_name column
df_merged = df_merged.dropna(subset=['file_name'])
print(f"Number of rows after dropping NaN values: {len(df_merged)}")
num_unique_writers = df_merged['writer'].nunique()
print(f"Number of unique writers in the dataset: {num_unique_writers}")

Number of NaN values in file_name column: 8
Number of rows after dropping NaN values: 12949
Number of unique writers in the dataset: 216


In [36]:
#split in train, validation and test sets
#icdar is 475 writers-> 282, 71, 122 -> i can keep comparable numbers
#iam is 217 writers (but i have more pages per writer), i keep 40 % for val and test and 60% of these for testing
n_train = int(num_unique_writers * 0.6)
n_val_test = num_unique_writers - n_train
n_val = int(n_val_test * 0.4)
n_test = n_val_test - n_val
print(n_train, n_val, n_test)
#get unique writers
unique_writers = df_merged['writer'].unique()
#shuffle the unique writers
np.random.shuffle(unique_writers)
#split the unique writers into train, val and test
train_writers = unique_writers[:n_train]
val_writers = unique_writers[n_train:n_train+n_val]
test_writers = unique_writers[n_train+n_val:]

train_df = df_merged[df_merged['writer'].isin(train_writers)]
val_df = df_merged[df_merged['writer'].isin(val_writers)]
test_df = df_merged[df_merged['writer'].isin(test_writers)]
print(len(train_df), len(val_df), len(test_df))


129 34 53
7882 1968 3099


# save

In [37]:
train_df.to_csv(os.path.join(output_path, f"{dataset_name}_train_df.csv"), index=False)
val_df.to_csv(os.path.join(output_path, f"{dataset_name}_public_df.csv"), index=False)
test_df.to_csv(os.path.join(output_path, f"{dataset_name}_private_df.csv"), index=False)

In [38]:
#load the train, val and test dataframes and count the number of unique writers in each
train_df = pd.read_csv(os.path.join(output_path, f"{dataset_name}_train_df.csv"))
val_df = pd.read_csv(os.path.join(output_path, f"{dataset_name}_public_df.csv"))
test_df = pd.read_csv(os.path.join(output_path, f"{dataset_name}_private_df.csv"))
print(f"Number of unique writers in train set: {len(train_df['writer'].unique())}")
print(f"Number of unique writers in validation set: {len(val_df['writer'].unique())}")
print(f"Number of unique writers in test set: {len(test_df['writer'].unique())}")

Number of unique writers in train set: 129
Number of unique writers in validation set: 34
Number of unique writers in test set: 53


# Easy access

In [4]:
# functions
def get_all_file_paths(directory):
    file_paths = []
    for root, _, files in os.walk(directory):
        for file in files:
            file_paths.append(os.path.join(root, file))
    return file_paths